<a href="https://colab.research.google.com/github/lennartvoelz/fine_tune_hf/blob/main/test_lora_merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2

In [2]:
!pip install modelscope
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 39.8 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
from datasets import load_dataset

In [4]:
import json

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/data/"

raw_data = []
with open(SAVE_DIR+"examples.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

def format_cactus_call(expected_calls):
    """Replicates the Cactus C++ expected output format"""
    if not expected_calls:
        return ""

    call = expected_calls[0]
    result = f"<start_function_call>call:{call['name']}{{"

    args = []
    for k, v in call["arguments"].items():
        if isinstance(v, str):
            args.append(f"{k}:<escape>{v}<escape>")
        else:
            args.append(f"{k}:{v}")

    result += ",".join(args) + "}<end_function_call>"
    return result

def format_cactus_tools(tools_list):
    """Replicates the Cactus format_tools() C++ function"""
    result = ""
    for t in tools_list:
        result += "<start_function_declaration>"
        result += f"declaration:{t['name']}{{"

        desc = t.get('description', '')
        result += f"description:<escape>{desc}<escape>"

        if "parameters" in t and "properties" in t["parameters"]:
            result += ",parameters:{properties:{"
            props = []
            for k, v in t["parameters"]["properties"].items():
                p_desc = v.get('description', '')
                p_type = v.get('type', 'string').upper()
                prop_str = f"{k}:{{description:<escape>{p_desc}<escape>,type:<escape>{p_type}<escape>}}"
                props.append(prop_str)
            result += ",".join(props) + "}"

            if "required" in t["parameters"]:
                reqs = [f"<escape>{r}<escape>" for r in t["parameters"]["required"]]
                result += f",required:[{','.join(reqs)}]"

            t_type = t['parameters'].get('type', 'object').upper()
            result += f",type:<escape>{t_type}<escape>}}"

        result += "}<end_function_declaration>"
    return result

def format_tools_for_prompt(example):
    user_msg = example["messages"][0]["content"]
    expected_calls = example.get("expected_calls", [])
    tools_list = example.get("tools", [])

    # --- 1. BUILD THE SYSTEM PROMPT (EXACTLY AS C++ DOES) ---
    system_content = "You are a helpful assistant that can use tools.\n"

    if tools_list:
        tools_json = format_cactus_tools(tools_list)
        system_content += "You are a model that can do function calling with the following functions."
        system_content += tools_json
        system_content += "\n\nWhen you decide to call a function, output it in this exact format:\n"
        system_content += "<start_function_call>call:function_name{arg1:<escape>value1<escape>,arg2:<escape>value2<escape>}<end_function_call>"

    # --- 2. BUILD THE ASSISTANT CONTENT (EXACTLY AS C++ EXPECTS) ---
    if expected_calls:
        assistant_content = format_cactus_call(expected_calls)
    else:
        assistant_content = example.get("expected_response", "")

    # --- 3. CONSTRUCT THE FULL CHAT STRING MANUALLY (BYPASSING JINJA/HF) ---
    # We build the exact string Gemma needs without apply_chat_template

    chat_str = (
        f"<start_of_turn>developer\n"
        f"{system_content}<end_of_turn>\n"
        f"<start_of_turn>user\n"
        f"{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{assistant_content}"
    )

    # Notice we DO NOT add a final <end_of_turn>\n here because we want the model
    # to learn to generate it after outputting the function call!

    return {
        "text": chat_str
    }

chat_data = []

for example in raw_data:
    formatted_example = format_tools_for_prompt(example)
    chat_data.append(formatted_example)

with open("chat_ready.jsonl", "w", encoding="utf-8") as f:
    for ex in chat_data:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
dataset = load_dataset("json", data_files="chat_ready.jsonl", split="train")

print(dataset[0])

In [ ]:
from peft import PeftModel, PeftConfig
from unsloth import FastLanguageModel
import torch

base_model, base_tokenizer = FastLanguageModel.from_pretrained("google/functiongemma-270m-it")
peft_model = PeftModel.from_pretrained(base_model, SAVE_DIR+"lora_dir")
merged_model = peft_model.merge_and_unload()

In [ ]:
# 1. Grab the raw example from your data
example = raw_data[0]
user_msg = example["messages"][0]["content"]
tools_list = example.get("tools", [])
eos_token_id = base_tokenizer.eos_token_id
end_call_token_id = base_tokenizer.convert_tokens_to_ids("<end_function_call>")

# 2. Build the exact Cactus System Prompt (reusing our logic from before)
system_content = "You are a helpful assistant that can use tools.\n"
if tools_list:
    tools_json = format_cactus_tools(tools_list) # Make sure this function is defined!
    system_content += "You are a model that can do function calling with the following functions."
    system_content += tools_json
    system_content += "\n\nWhen you decide to call a function, output it in this exact format:\n"
    system_content += "<start_function_call>call:function_name{arg1:<escape>value1<escape>,arg2:<escape>value2<escape>}<end_function_call>"

# 3. Manually construct the prompt string for generation
# Notice we add "<start_of_turn>model\n" at the end, which acts as the "add_generation_prompt=True"
text = (
    f"<start_of_turn>developer\n"
    f"{system_content}<end_of_turn>\n"
    f"<start_of_turn>user\n"
    f"{user_msg}<end_of_turn>\n"
    f"<start_of_turn>model\n"
)

# 4. Tokenize and prepare for Unsloth / HF Generation
inputs = base_tokenizer(text, return_tensors="pt").to("cuda")

from transformers import TextStreamer
# skip_prompt=True keeps the output clean so you only see the generated function call
streamer = TextStreamer(base_tokenizer, skip_prompt=False)

# 5. Generate with strict deterministic settings
_ = merged_model.generate(
    **inputs,
    max_new_tokens=256,
    streamer=streamer,
    do_sample=False,   # Greedy decoding is required for JSON/Function calls
    temperature=0.0,
    eos_token_id=[eos_token_id, end_call_token_id]
)